# Predex — Pokémon Battle Prediction (FDS 2026)

This notebook reproduces our **three final submissions**. All the code lives in our GitHub repository: the notebook clones it, builds the features from the competition data and trains the final models.

**Pipeline in short.** For every battle we compute features from Player 1's team, Player 2's lead and the first 30 turns (HP, KOs, status, boosts, moves used, species still alive, entry hazards, type matchups). On top of them we add a *player skill* estimate: the same 6-Pokémon teams appear in many battles, so the win rate of a team in the **training** battles is informative. The models are LightGBM and logistic regression. Every decision was taken with the same 10-fold stratified cross-validation (`data/train_folds.csv`), and checked on a second 10-fold split (seed 2026).

| Final submission | Model | CV accuracy (team folds / second split) | CV log-loss |
|---|---|---|---|
| 1 | Blend: 0.70 LightGBM + 0.15 logistic regression + 0.15 logistic regression (second feature set) | 0.7044 / 0.7015 | 0.5626 / 0.5613 |
| 2 | LightGBM only (average of 5 seeds) | 0.7013 / 0.7006 | 0.5652 / 0.5634 |
| 3 | Same blend as 1, **without** the player-skill features | 0.6970 / 0.6989 | 0.5704 / 0.5687 |

The scripts compute these numbers when run without `--no-cv` (slower); here we only train the final models.

## 1. Setup

Clone the repository and copy the competition files where the code expects them (`raw_data/`). The type chart is needed too: without it the type-matchup features fall back to neutral values.

In [ ]:
import os, shutil, subprocess

REPO_URL = "https://github.com/Ma77eoma770/Predex.git"
INPUT = "/kaggle/input/competitions/fds-pokemon-battles-prediction-2026"
WORK = "/kaggle/working"

if not os.path.exists(f"{WORK}/Predex"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, f"{WORK}/Predex"], check=True)
os.chdir(f"{WORK}/Predex")

os.makedirs("raw_data", exist_ok=True)
for name in ["train.jsonl", "test.jsonl", "sample_submission.csv", "type_chart_gen2.csv"]:
    shutil.copy(f"{INPUT}/{name}", f"raw_data/{name}")
print(sorted(os.listdir("raw_data")))

We install the two libraries that decide the predictions with the same versions we used (see `requirements_final.txt`), so that the outputs match the files we submitted.

In [ ]:
!pip install -q lightgbm==4.7.0 scikit-learn==1.9.1
import lightgbm, sklearn
print("lightgbm", lightgbm.__version__, "| scikit-learn", sklearn.__version__)

## 2. Feature store

`src/features/registry.py` runs every feature module in `src/features/modules/` on each battle and saves one table for the train set and one for the test set (794 columns). Each feature depends only on the battle itself, or on statistics computed from `train.jsonl` without labels (for example how often a species appears in the training teams, in `src/features/data_priors.py`). No external data is used.

`data/pokedex.json` (types and base stats per species) is rebuilt first from `train.jsonl` only, with `src/features/build_pokedex.py`.

In [ ]:
!python src/features/build_pokedex.py
!python src/features/registry.py --force

## 3. Training the final models

`src/models/train_19_final_max.py` trains three models on all 10,000 training battles:
* **LightGBM**, average of 5 seeds, hyperparameters tuned with Optuna on the cross-validated log-loss. Features: 786 team features + 33 field/threat features (spikes, rapid spin, screens, boosts at turn 30) + 5 player-skill features.
* **Logistic regression** on the 786 features + player skill (standardised, collinear features removed).
* **Logistic regression** on a smaller set of 122 features (material, status, moves, field, threat) + player skill.

Player skill = smoothed win rate of the same team in the training battles, `(wins + 2.5) / (games + 5)`, for Player 1 and, when all 6 Pokémon have been revealed, for Player 2. In cross-validation it is computed inside each fold, so a battle never sees its own result; on the training set used for the final model it is computed with an inner 5-fold split for the same reason.

The logistic regression on 122 features uses the feature set designed by our teammate Max (`src/features/max_features.py`).

In [ ]:
!python src/models/train_19_final_max.py --no-cv

### Submission 1 — blend with player skill

Average of the three models with weights 0.70 (LightGBM) / 0.15 / 0.15 (the two logistic regressions), fixed before looking at the results. The logistic regressions make different mistakes from LightGBM, so the average is more reliable than any single model; it is our best model in cross-validation.

In [ ]:
shutil.copy("outputs/submission_19_blend3.csv", f"{WORK}/final_1_blend3.csv")

### Submission 2 — LightGBM alone

The same LightGBM (average of 5 seeds) without the logistic regressions: our best single model.

In [ ]:
shutil.copy("outputs/submission_19_lgbm5_max.csv", f"{WORK}/final_2_lgbm5.csv")

## 4. Submission 3 — same blend without player skill

`src/models/train_20_noskill_clean.py` repeats submission 1 without the 5 player-skill features. It is our safety choice: if teams repeat less in the private test set than in the training set, the skill features would help less, while this model does not depend on them. In cross-validation it is about 0.5 points below submission 1.

In [ ]:
!python src/models/train_20_noskill_clean.py --no-cv
shutil.copy("outputs/submission_20_blend3_noskill_clean.csv", f"{WORK}/final_3_noskill.csv")

## 5. Checks

Every file must contain one prediction for each test battle, in the format of `sample_submission.csv`.

In [ ]:
import pandas as pd

sample = pd.read_csv("raw_data/sample_submission.csv")
for name in ["final_1_blend3.csv", "final_2_lgbm5.csv", "final_3_noskill.csv"]:
    sub = pd.read_csv(f"{WORK}/{name}")
    assert list(sub.columns) == ["battle_id", "player_won"]
    assert len(sub) == len(sample) and set(sub.battle_id) == set(sample.battle_id)
    print(f"{name:<22} rows {len(sub)} | predicted P1 wins {sub.player_won.mean():.1%}")